# 🗂️ Notebook 2: Reddit — Data Model & APIs


## 🎯 Learning objectives

- Design the core tables — users, subreddits, posts, votes, comments — and understand
  *why* we denormalise vote counts onto `posts`.
- Run a tiny SQLite schema and CRUD flow end-to-end (no external services needed).
- Model vote semantics correctly: **+1 / 0 / -1** with idempotent toggling.
- Model threaded comments two ways: **adjacency list** (`parent_id`) and **materialized path**,
  and see when to use each.
- Define the user-facing HTTP API with Pydantic request/response models.


## 🛠️ Setup

```bash
cd 06-system-designs/reddit
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Schema overview

```sql
users        (id, name)
subreddits   (id, name, created_at)
memberships  (user_id, subreddit_id)                       -- many-to-many

posts        (id, subreddit_id, author_id, title, body,
              created_at,
              ups, downs, hot_score)                       -- ← denormalised

comments     (id, post_id, parent_id, author_id, body,
              created_at, path)                            -- adjacency + materialized path

votes        (user_id, target_kind, target_id, value)      -- value ∈ {-1, +1}
             PRIMARY KEY (user_id, target_kind, target_id) -- idempotent
```

### Why denormalise `ups`, `downs`, `hot_score` onto `posts`?

A feed request needs to sort ~1 k posts by hot_score. If `ups` lived only in `votes`, every
feed read would have to `SELECT SUM(value) FROM votes GROUP BY post_id` — a table scan per
request. Instead we **keep a cached count on the post row** and update it asynchronously.

We still keep the raw `votes` table so that:

- We can answer *"did I vote on this?"* (the UI needs the arrow highlighted).
- A user can toggle their vote (see §3 below).
- We can recount from scratch if the cache drifts.


## 2. Let's build it — SQLite in-memory so it's fully runnable


In [ ]:
import sqlite3, textwrap
from datetime import datetime

# In-memory DB — throw-away, perfect for demos.
db = sqlite3.connect(":memory:")
db.row_factory = sqlite3.Row
db.executescript(textwrap.dedent('''
    CREATE TABLE users       (id INTEGER PRIMARY KEY, name TEXT UNIQUE);
    CREATE TABLE subreddits  (id INTEGER PRIMARY KEY, name TEXT UNIQUE, created_at TEXT);
    CREATE TABLE posts (
        id INTEGER PRIMARY KEY,
        subreddit_id INTEGER,
        author_id INTEGER,
        title TEXT,
        body TEXT,
        created_at TEXT,
        ups   INTEGER DEFAULT 0,
        downs INTEGER DEFAULT 0,
        hot_score REAL DEFAULT 0
    );
    CREATE TABLE comments (
        id INTEGER PRIMARY KEY,
        post_id INTEGER,
        parent_id INTEGER,
        author_id INTEGER,
        body TEXT,
        created_at TEXT,
        path TEXT            -- materialized path e.g. '/7/12/' (see §5)
    );
    CREATE INDEX ix_comments_path ON comments(path);

    CREATE TABLE votes (
        user_id     INTEGER,
        target_kind TEXT,     -- 'post' or 'comment'
        target_id   INTEGER,
        value       INTEGER,  -- -1 or +1
        PRIMARY KEY (user_id, target_kind, target_id)
    );
'''))

# Seed some data
db.executemany("INSERT INTO users(name) VALUES (?)",
               [("alice",), ("bob",), ("carol",), ("dave",)])
db.execute("INSERT INTO subreddits(name, created_at) VALUES ('python', ?)",
           (datetime.utcnow().isoformat(),))
db.execute("INSERT INTO posts(subreddit_id, author_id, title, body, created_at) "
           "VALUES (1, 1, 'Hello r/python', 'first post', ?)",
           (datetime.utcnow().isoformat(),))
db.commit()

for row in db.execute("SELECT id, title, ups, downs FROM posts"):
    print(dict(row))


## 3. Voting — the tricky part

A vote is not just "insert row". It must be **idempotent**: clicking the up-arrow twice removes
the vote; clicking up then down replaces it. So the operation is:

```
cast_vote(user, target, new_value ∈ {-1, 0, +1}):
    old_value = SELECT value FROM votes WHERE (user, target)
    if old_value == new_value:              # already in that state — no-op (or un-vote)
        delete, adjust counters by -old_value
    elif new_value == 0:
        delete, adjust counters by -old_value
    else:
        upsert, adjust counters by (new_value - old_value)
```

The **counter delta** is always `new − old`, which is at most 2 (swing from -1 to +1). That's
how we keep `posts.ups / posts.downs` in sync cheaply.


In [ ]:
def cast_vote(user_id: int, post_id: int, new_value: int) -> None:
    assert new_value in (-1, 0, 1), "direction must be -1, 0, or +1"

    row = db.execute(
        "SELECT value FROM votes WHERE user_id=? AND target_kind='post' AND target_id=?",
        (user_id, post_id),
    ).fetchone()
    old_value = row["value"] if row else 0

    if old_value == new_value:
        return  # no change

    # Update the votes table
    if new_value == 0:
        db.execute("DELETE FROM votes WHERE user_id=? AND target_kind='post' AND target_id=?",
                   (user_id, post_id))
    else:
        db.execute('''INSERT INTO votes(user_id, target_kind, target_id, value)
                      VALUES (?, 'post', ?, ?)
                      ON CONFLICT(user_id, target_kind, target_id)
                      DO UPDATE SET value = excluded.value''',
                   (user_id, post_id, new_value))

    # Adjust denormalised counters
    up_delta   = max(new_value, 0) - max(old_value, 0)     # +1 enters/leaves "ups"
    down_delta = max(-new_value, 0) - max(-old_value, 0)   # -1 enters/leaves "downs"
    db.execute("UPDATE posts SET ups = ups + ?, downs = downs + ? WHERE id=?",
               (up_delta, down_delta, post_id))
    db.commit()

def show(post_id):
    p = db.execute("SELECT ups, downs FROM posts WHERE id=?", (post_id,)).fetchone()
    print(f"post {post_id}: ups={p['ups']} downs={p['downs']}")

cast_vote(user_id=2, post_id=1, new_value=+1);  show(1)
cast_vote(user_id=3, post_id=1, new_value=+1);  show(1)
cast_vote(user_id=4, post_id=1, new_value=-1);  show(1)
cast_vote(user_id=2, post_id=1, new_value=+1);  show(1)   # no-op
cast_vote(user_id=2, post_id=1, new_value=-1);  show(1)   # swing +1 → -1
cast_vote(user_id=2, post_id=1, new_value= 0);  show(1)   # un-vote


Things to notice:

- The counters in `posts` stay exactly consistent with the sum over `votes`.
- Repeated clicks don't double-count: we always compute a **delta**.
- In production you would also emit a vote event to Kafka here, so the **ranking worker** sees it.


## 4. Denormalised vs live-aggregate — measure the difference

Let's load some fake votes and compare: reading a feed of 1 000 posts where we either
(a) store `ups` on the post row, or (b) compute it from `votes` on every request.


In [ ]:
import random, time
random.seed(0)

# Seed 1 000 posts
db.executemany(
    "INSERT INTO posts(subreddit_id, author_id, title, body, created_at) VALUES (1,1,?,?,?)",
    [(f"post #{i}", "body", datetime.utcnow().isoformat()) for i in range(1_000)],
)
# Seed ~50 000 votes
vote_rows = []
for pid in range(2, 1002):
    for uid in random.sample(range(1, 51), random.randint(10, 50)):
        vote_rows.append((uid, "post", pid, random.choice([-1, 1])))
db.executemany("INSERT OR IGNORE INTO votes VALUES (?,?,?,?)", vote_rows)
# Keep the denormalised counters in sync (one-off backfill)
db.executescript('''
    UPDATE posts SET
      ups   = (SELECT COALESCE(SUM(CASE WHEN value=1  THEN 1 ELSE 0 END),0)
               FROM votes WHERE target_kind='post' AND target_id=posts.id),
      downs = (SELECT COALESCE(SUM(CASE WHEN value=-1 THEN 1 ELSE 0 END),0)
               FROM votes WHERE target_kind='post' AND target_id=posts.id);
''')
db.commit()

# (a) denormalised read
t0 = time.time()
rows = db.execute("SELECT id, title, ups - downs AS score FROM posts ORDER BY score DESC LIMIT 25").fetchall()
denorm_ms = (time.time() - t0) * 1000

# (b) live aggregate read
t0 = time.time()
rows2 = db.execute('''
    SELECT p.id, p.title,
           COALESCE(SUM(v.value), 0) AS score
    FROM posts p LEFT JOIN votes v
      ON v.target_kind='post' AND v.target_id=p.id
    GROUP BY p.id
    ORDER BY score DESC
    LIMIT 25
''').fetchall()
live_ms = (time.time() - t0) * 1000

print(f"denormalised read: {denorm_ms:6.2f} ms")
print(f"live aggregate   : {live_ms:6.2f} ms   (same answer, slower)")
print(f"slowdown         : {live_ms/denorm_ms:4.1f}x — and it gets worse with scale")


With only 50 k votes the aggregate is already noticeably slower. At Reddit scale
(**billions** of votes) the aggregate is impossible; the denormalised count is the only path.

The downside is that we can drift out of sync (e.g. if the counter-updater crashes mid-way).
A **nightly reconciliation job** that recomputes from `votes` fixes any drift.


## 5. Threaded comments — two storage patterns

Reddit comments form a **tree** per post:

```
Post
├── alice: great post!
│   └── bob: +1
│        └── carol: me too
└── dave: I disagree
    └── alice: why?
```

### 5a. Adjacency list — `parent_id` only

Simple: each row points to its parent. But **"give me the whole subtree rooted at comment X"**
requires recursion (N queries or a recursive CTE).

### 5b. Materialized path — store `/ancestor_ids/`

We also store a path like `/7/12/18/` which lets us fetch a subtree with one index lookup:

```sql
SELECT * FROM comments WHERE path LIKE '/7/12/%' ORDER BY path;
```

Indexed prefix lookup → extremely fast.


In [ ]:
# A tiny helper that writes both parent_id AND path.
def add_comment(post_id: int, parent_id: int | None, author_id: int, body: str) -> int:
    cur = db.execute(
        "INSERT INTO comments(post_id, parent_id, author_id, body, created_at) VALUES (?,?,?,?,?)",
        (post_id, parent_id, author_id, body, datetime.utcnow().isoformat()),
    )
    cid = cur.lastrowid
    if parent_id is None:
        path = f"/{cid}/"
    else:
        parent_path = db.execute("SELECT path FROM comments WHERE id=?", (parent_id,)).fetchone()["path"]
        path = f"{parent_path}{cid}/"
    db.execute("UPDATE comments SET path=? WHERE id=?", (path, cid))
    db.commit()
    return cid

# Build the example tree on post #1
c1 = add_comment(1, None, 1, "great post!")
c2 = add_comment(1, c1,   2, "+1")
c3 = add_comment(1, c2,   3, "me too")
c4 = add_comment(1, None, 4, "I disagree")
c5 = add_comment(1, c4,   1, "why?")

for r in db.execute("SELECT id, parent_id, path, body FROM comments WHERE post_id=1 ORDER BY path"):
    indent = "  " * (r["path"].count("/") - 2)
    print(f"{r['path']:<18} {indent}{r['body']}")


In [ ]:
# Fetch *just the subtree rooted at c1*, using the materialized path — one indexed query.
root = db.execute("SELECT path FROM comments WHERE id=?", (c1,)).fetchone()["path"]
for r in db.execute("SELECT body, path FROM comments WHERE path LIKE ? ORDER BY path", (root + "%",)):
    depth = r["path"].count("/") - root.count("/")
    print("  " * depth, "-", r["body"])


### When to use which?

| Storage | Best for | Weak at |
|---|---|---|
| Adjacency list (`parent_id`) only | single-level reads; easy writes | deep subtree reads (recursion) |
| Materialized path | **fast subtree reads** (Reddit-style "load this thread") | moves/rewrites (path of every descendant must be updated) |
| **Closure table** (separate ancestor/descendant rows) | arbitrary tree queries | more storage |

Reddit actually uses a mix: `parent_id` + precomputed sorted comment id lists per `(post, sort)`
stored in a cache. The point is that a *naïve* recursive query is never on the hot path.


## 6. The HTTP API (Pydantic models)

Below we declare the request/response contracts. In a real service these would live in a FastAPI
app; here we just print the JSON so the cell is self-contained.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
from datetime import datetime

# ---------- Requests ----------
class CreatePostRequest(BaseModel):
    title: str = Field(min_length=1, max_length=300)
    body: Optional[str] = None
    url: Optional[str] = None

class VoteRequest(BaseModel):
    # -1 = downvote, 0 = remove vote, +1 = upvote
    dir: Literal[-1, 0, 1]

class CreateCommentRequest(BaseModel):
    body: str
    parent_id: Optional[int] = None   # None → top-level comment

# ---------- Responses ----------
class PostResponse(BaseModel):
    id: int
    subreddit: str
    author: str
    title: str
    body: Optional[str] = None
    ups: int
    downs: int
    score: int
    created_at: datetime
    my_vote: Literal[-1, 0, 1] = 0   # what the *current user* voted — for UI highlight

class CommentResponse(BaseModel):
    id: int
    parent_id: Optional[int]
    author: str
    body: str
    score: int
    created_at: datetime
    depth: int                       # how deep in the tree — used for indentation

# Show a couple of examples
print(VoteRequest(dir=1).model_dump_json())
print(CreatePostRequest(title="Hello", body="world").model_dump_json())
print(CommentResponse(id=1, parent_id=None, author="alice", body="hi",
                      score=5, created_at=datetime.utcnow(), depth=0)
      .model_dump_json())


### Key endpoints

```http
POST /r/{name}/posts             # CreatePostRequest  -> PostResponse
POST /posts/{id}/vote            # VoteRequest        -> {ok, score}
GET  /r/{name}?sort=hot&after=…  # cursor-paginated   -> list[PostResponse]
GET  /r/all?sort=hot             # merges top-K from subscribed subs
POST /posts/{id}/comments        # CreateCommentRequest -> CommentResponse
GET  /posts/{id}/comments?limit=200&more_children=…  # paginated tree
```

Two things to note:

- Feeds use **cursor pagination** (`?after=t3_abcdef`) not `offset` — offsets get wrong as
  new posts arrive at the top.
- The comment endpoint returns a *flat* list whose `depth` field lets the client render the
  tree; no recursion in the API.


## 📚 Summary

- **Denormalise counts** onto `posts` — live aggregation from `votes` doesn't scale.
- **Votes are deltas**, not inserts. Always compute `new_value − old_value`.
- **Threaded comments** need a materialized path (or closure table) to fetch subtrees cheaply.
- Pydantic + cursor pagination keep the API small and stable.

### 💡 Interview tips

- When the interviewer asks about schema, *always* mention denormalisation and why.
- Saying "vote is idempotent, driven by delta" sounds senior.
- Mention the **reconciliation job** that recomputes denormalised counters nightly — shows
  you understand the cost of denormalisation.

### Next up

In **Notebook 3** we dive into the three classic Reddit deep-dives with *bad → better → best*
code: the Hot ranking formula, sharded counters for vote hot-spots, and serving comment trees.
